# Lab 7.2 &mdash; Build the Tracer

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 7 &mdash; Multi-Agent System Evaluation**

### What you'll do
- Record spans with parent links, so the nesting survives
- Separate a span's own time from the time it spent inside its children
- Roll cost and latency up by agent, and find that they name different villains
- Then look at LangFuse and recognise every field

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **About forty lines.** Once you have written a span tree by hand, a tracing product
> is a UI over something you understand rather than a black box you configure.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-7-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

## Concept

A log is a list. A trace is a tree, and the tree is the information: it is the only thing that
lets you say the 3.9 seconds of retrieval *belonged to* the policy agent rather than merely
happening near it.

The clock here is synthetic and advanced by hand, so every number in this lab is exact. A real
tracer swaps `tick()` for `time.perf_counter()` and changes nothing else.

## Section 1 &mdash; Spans, and what contains what

One list of spans, each knowing its parent. That is the whole data structure.

In [ ]:
class Tracer:
    """A span tree with a hand-cranked clock, so the numbers are exact."""

    def __init__(self):
        self.spans, self._stack, self.clock = [], [], 0.0

    def tick(self, seconds: float):
        """Advance the synthetic clock. A real tracer reads time.perf_counter() instead."""
        self.clock += seconds

    def start(self, name: str, kind: str = "agent") -> int:
        sid = len(self.spans)
        parent = self._stack[-1] if self._stack else None
        self.spans.append({"id": sid, "parent": parent, "name": name, "kind": kind,
                           "t0": self.clock, "t1": None, "tokens": 0})
        self._stack.append(sid)
        return sid

    def end(self, sid: int, tokens: int = 0):
        self.spans[sid]["t1"] = self.clock
        self.spans[sid]["tokens"] = tokens
        self._stack.pop()


def sample_run() -> Tracer:
    """The run from the deck: supervisor, ledger, policy (containing retrieval), writer."""
    t = Tracer()
    run = t.start("run", "run")
    s = t.start("supervisor");            t.tick(0.4);  t.end(s, tokens=120)
    l = t.start("ledger");                t.tick(0.1)
    lt = t.start("lookup_payment", "tool"); t.tick(0.8); t.end(lt)
    t.end(l, tokens=380)
    p = t.start("policy");                t.tick(0.2)
    r = t.start("retrieval", "tool");     t.tick(3.9);  t.end(r, tokens=12)
    t.end(p, tokens=420)
    w = t.start("writer");                t.tick(1.7);  t.end(w, tokens=2260)
    t.end(run)
    return t

In [ ]:
# --- Self-check: Section 1
def spans():
    return sample_run().spans

def by_name(name):
    return next(s for s in spans() if s["name"] == name)

check("every span was closed",
      lambda: all(s["t1"] is not None for s in spans()))
check("the root has no parent",
      lambda: by_name("run")["parent"] is None)
check("the supervisor's parent is the run",
      lambda: by_name("supervisor")["parent"] == by_name("run")["id"])
check("RETRIEVAL'S PARENT IS THE POLICY AGENT, not the run",
      lambda: by_name("retrieval")["parent"] == by_name("policy")["id"],
      "this one link is the difference between a trace and a log")
check("the tool call sits inside the ledger agent",
      lambda: by_name("lookup_payment")["parent"] == by_name("ledger")["id"])
check("the stack is empty when the run finishes",
      lambda: sample_run()._stack == [],
      "an unbalanced start/end leaves a span open and every later parent wrong")
check("the whole run took 7.1 seconds",
      lambda: abs(by_name("run")["t1"] - by_name("run")["t0"] - 7.1) < 1e-9)

## Section 2 &mdash; Its own time, and its children's

The policy agent took 4.1 seconds. It *spent* 0.2 of them. Attribution needs the difference.

In [ ]:
def total_time(spans: list, sid: int) -> float:
    """Wall clock from the moment this span opened to the moment it closed."""
    s = spans[sid]
    return round(s["t1"] - s["t0"], 6)


def children(spans: list, sid: int) -> list:
    return [s for s in spans if s["parent"] == sid]


def self_time(spans: list, sid: int) -> float:
    """Time inside this span that was NOT spent inside one of its children."""
    return round(total_time(spans, sid)
                 - sum(total_time(spans, c["id"]) for c in children(spans, sid)), 6)


def tokens_including_children(spans: list, sid: int) -> int:
    """What this span cost, counting everything that ran inside it."""
    return spans[sid]["tokens"] + sum(tokens_including_children(spans, c["id"])
                                      for c in children(spans, sid))

In [ ]:
# --- Self-check: Section 2
def sp():
    return spans()

check("the policy agent's total is 4.1 seconds",
      lambda: abs(total_time(sp(), by_name("policy")["id"]) - 4.1) < 1e-9)
check("but it only spent 0.2 of them itself",
      lambda: abs(self_time(sp(), by_name("policy")["id"]) - 0.2) < 1e-9,
      "the policy agent is not slow -- it contains something slow, and only the tree says so")
check("retrieval has no children, so its self time is its total",
      lambda: self_time(sp(), by_name("retrieval")["id"])
              == total_time(sp(), by_name("retrieval")["id"]))
check("the self times of every span add up to the whole run",
      lambda: abs(sum(self_time(sp(), s["id"]) for s in sp())
                  - total_time(sp(), by_name("run")["id"])) < 1e-9,
      "if this does not hold, the tree is wrong and every attribution built on it is wrong")
check("the run's token total includes everything beneath it",
      lambda: tokens_including_children(sp(), by_name("run")["id"]) == 3192)
check("the policy agent is charged for its retrieval",
      lambda: tokens_including_children(sp(), by_name("policy")["id"]) == 432)
check("a leaf's inclusive tokens are just its own",
      lambda: tokens_including_children(sp(), by_name("writer")["id"]) == 2260)

## Section 3 &mdash; Two different villains

Now roll it up and rank it twice: once by time, once by tokens.

In [ ]:
def breakdown(spans: list) -> list:
    """One row per span: what it spent itself, and what it cost inclusive of children."""
    return [{"name": s["name"], "kind": s["kind"],
             "self_s": self_time(spans, s["id"]),
             "total_s": total_time(spans, s["id"]),
             "tokens": s["tokens"]}
            for s in spans if s["parent"] is not None]


def worst_by(spans: list, key: str) -> str:
    """The span that dominates one axis."""
    return max(breakdown(spans), key=lambda r: r[key])["name"]


def share(spans: list, name: str, key: str) -> float:
    rows = breakdown(spans)
    total = sum(r[key] for r in rows)
    row = next(r for r in rows if r["name"] == name)
    return row[key] / total if total else 0.0


def _report():
    rows = sorted(breakdown(spans()), key=lambda r: -r["self_s"])
    print(f"  {'span':18}{'kind':8}{'self s':>9}{'total s':>9}{'tokens':>9}")
    print("  " + "-" * 54)
    for r in rows:
        print(f"  {r['name']:18}{r['kind']:8}{r['self_s']:>9.1f}{r['total_s']:>9.1f}"
              f"{r['tokens']:>9}")
    print()
    print(f"  slowest step : {worst_by(spans(), 'self_s')}  "
          f"({share(spans(), worst_by(spans(), 'self_s'), 'self_s'):.0%} of the wall clock)")
    print(f"  dearest step : {worst_by(spans(), 'tokens')}  "
          f"({share(spans(), worst_by(spans(), 'tokens'), 'tokens'):.0%} of the bill)")
guard(_report)

In [ ]:
# --- Self-check: Section 3
check("the slowest step is retrieval",
      lambda: worst_by(spans(), "self_s") == "retrieval")
check("the dearest step is the writer",
      lambda: worst_by(spans(), "tokens") == "writer")
check("THEY ARE NOT THE SAME STEP",
      lambda: worst_by(spans(), "self_s") != worst_by(spans(), "tokens"),
      "optimise the biggest number on the wrong axis and you work hard and save nothing")
check("retrieval is more than half the wall clock",
      lambda: share(spans(), "retrieval", "self_s") > 0.5)
check("and almost none of the bill",
      lambda: share(spans(), "retrieval", "tokens") < 0.01)
check("the writer is most of the bill",
      lambda: share(spans(), "writer", "tokens") > 0.7)
check("the root is excluded from the breakdown, or everything double-counts",
      lambda: all(r["name"] != "run" for r in breakdown(spans())))

## Run it for real &mdash; LangFuse

Every field you just built has a name in a tracing product: your span is a *span*, a model call is
a *generation*, the whole thing is a *trace*, and `parent` is what draws the tree.

This cell sends the same run to LangFuse if it is configured. It reads its settings from the
environment and hardcodes nothing.

In [ ]:
def send_to_langfuse():
    host = os.environ.get("LANGFUSE_HOST")
    pk   = os.environ.get("LANGFUSE_PUBLIC_KEY")
    sk   = os.environ.get("LANGFUSE_SECRET_KEY")
    if not (host and pk and sk):
        print("LangFuse is not configured in this sandbox. To point at one, set:")
        print("  export LANGFUSE_HOST=...        # the base URL")
        print("  export LANGFUSE_PUBLIC_KEY=pk-lf-...")
        print("  export LANGFUSE_SECRET_KEY=sk-lf-...")
        print()
        print("Nothing above needed it. The span tree you built carries the same information,")
        print("and the mapping is one line per field:")
        for ours, theirs in (("run span", "trace"), ("agent span", "span"),
                             ("a model call", "generation"), ("parent", "the tree itself"),
                             ("tokens", "usage"), ("your assertions", "scores")):
            print(f"    {ours:16} -> {theirs}")
        return
    from langfuse import Langfuse
    client = Langfuse(host=host, public_key=pk, secret_key=sk)
    t = sample_run()
    trace = client.trace(name="module-7-sample-run")
    for s in t.spans:
        if s["parent"] is None:
            continue
        trace.span(name=s["name"], start_time=None, end_time=None,
                   metadata={"self_s": self_time(t.spans, s["id"]),
                             "total_s": total_time(t.spans, s["id"]),
                             "tokens": s["tokens"], "kind": s["kind"]})
    client.flush()
    print(f"sent one trace to {host}")

guard(send_to_langfuse)

### Read it

If LangFuse is not wired up in your sandbox, you have lost nothing today: the mapping printed
above is the whole of what a tracing product adds on top of what you just built, plus storage, a
UI and a place to attach scores.

That is worth having in production and it is not worth being mystified by. The thing to take away
is the shape &mdash; **spans with parents, timing, usage, and scores attached to a trace id** &mdash; because
every vendor implements that shape and you can now read any of them.

In [ ]:
score()

## Your turn

1. Swap `tick()` for `time.perf_counter()` and instrument one function you actually own. The class
   does not change; only the clock does.
2. `end()` pops the stack blindly, so an exception between `start` and `end` corrupts every later
   parent. Make `start` a context manager so the span closes even when the body raises.
3. Add a `status` to each span and set it on failure. Then count, across many runs, which span
   fails most often &mdash; that ranking is Lab 7.4.